# Four-threshold directional-change mechanics

This notebook adds one scale to the causal GBPUSD.r hierarchy:

    small  0.20%
    medium 0.40%
    large  0.60%
    macro  0.80%

Each threshold owns an independent directional-change clock. Configuration order is always small / medium / large / macro.

The objective is to learn the complete four-scale structural profile: broad agreement, coherent propagation fronts, fragmentation, adjacent contests, and nesting. No HMM, classifier, trading rule, or terminal-depth model is introduced.

In [1]:
from enum import StrEnum
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from dc_basics.core import dc_live_features, parse_dc_events

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
MEDIUM_THETA = 0.004
LARGE_THETA = 0.006
MACRO_THETA = 0.008
N_PRICES = 20_000
CASEBOOK_BARS_BEFORE = 120
CASEBOOK_BARS_AFTER = 120

if not 0.0 < SMALL_THETA < MEDIUM_THETA < LARGE_THETA < MACRO_THETA:
    raise ValueError("Thresholds must satisfy 0 < small < medium < large < macro.")

print(f"Symbol: {SYMBOL}")
print(
    f"Thresholds: {SMALL_THETA:.2%}, {MEDIUM_THETA:.2%}, "
    f"{LARGE_THETA:.2%}, {MACRO_THETA:.2%}"
)

Symbol: GBPUSD.r
Thresholds: 0.20%, 0.40%, 0.60%, 0.80%


In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"
    MACRO = "macro"


from dc_basics.enums import DcMode


class ModeConfiguration(StrEnum):
    UP_UP_UP_UP = "up/up/up/up"
    UP_UP_UP_DOWN = "up/up/up/down"
    UP_UP_DOWN_UP = "up/up/down/up"
    UP_UP_DOWN_DOWN = "up/up/down/down"
    UP_DOWN_UP_UP = "up/down/up/up"
    UP_DOWN_UP_DOWN = "up/down/up/down"
    UP_DOWN_DOWN_UP = "up/down/down/up"
    UP_DOWN_DOWN_DOWN = "up/down/down/down"
    DOWN_UP_UP_UP = "down/up/up/up"
    DOWN_UP_UP_DOWN = "down/up/up/down"
    DOWN_UP_DOWN_UP = "down/up/down/up"
    DOWN_UP_DOWN_DOWN = "down/up/down/down"
    DOWN_DOWN_UP_UP = "down/down/up/up"
    DOWN_DOWN_UP_DOWN = "down/down/up/down"
    DOWN_DOWN_DOWN_UP = "down/down/down/up"
    DOWN_DOWN_DOWN_DOWN = "down/down/down/down"


class HierarchyShape(StrEnum):
    BROAD_AGREEMENT = "broad_agreement"
    COUNTER_DEPTH_1 = "counter_depth_1"
    COUNTER_DEPTH_2 = "counter_depth_2"
    COUNTER_DEPTH_3 = "counter_depth_3"
    FRAGMENTED_2_BOUNDARIES = "fragmented_2_boundaries"
    FRAGMENTED_3_BOUNDARIES = "fragmented_3_boundaries"


from dc_basics.enums import AttemptOutcome


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE, Scale.MACRO)
SCALE_SETTINGS = {
    Scale.SMALL: {
        "prefix": "S", "theta": SMALL_THETA, "up_color": "#16a34a",
        "down_color": "#dc2626", "line_width": 0.8, "line_dash": "solid",
    },
    Scale.MEDIUM: {
        "prefix": "M", "theta": MEDIUM_THETA, "up_color": "#2563eb",
        "down_color": "#f59e0b", "line_width": 1.1, "line_dash": "dash",
    },
    Scale.LARGE: {
        "prefix": "L", "theta": LARGE_THETA, "up_color": "#7c3aed",
        "down_color": "#0f766e", "line_width": 1.6, "line_dash": "dot",
    },
    Scale.MACRO: {
        "prefix": "X", "theta": MACRO_THETA, "up_color": "#c026d3",
        "down_color": "#92400e", "line_width": 2.2, "line_dash": "longdash",
    },
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}
CONFIGURATION_ORDER = [configuration.value for configuration in ModeConfiguration]

## Four-scale reading contract

A configuration is the current causal mode of four independent clocks. For example:

    down / down / up / up

means that the disturbance has been confirmed through medium, while large and macro remain up.

A coherent propagation front has zero or one directional boundary across the ordered scales. Two or three boundaries indicate fragmentation:

    down / up / down / down   has two boundaries
    down / up / down / up     has three boundaries

A shape describes what is known now. It does not reveal the next DCC or the eventual propagation depth.

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no close column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


## Reconstruct four independent event clocks

Completed TMV, T, and R become known at the event's DCC confirmation time. Live mode is causal on every initialized bar.

In [4]:
from dc_basics.events import prepare_event_ledger


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    settings = SCALE_SETTINGS[scale]
    raw_events = parse_dc_events(prices=prices, theta=settings["theta"])
    ledger = prepare_event_ledger(raw_events, settings["prefix"], scale, settings["theta"])
    live = dc_live_features(prices, theta=settings["theta"])
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    event_lookup = ledger.set_index("dcc_idx")["event_id"]
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(event_lookup)

In [5]:
clock_records = []
for scale in SCALE_ORDER:
    ledger = event_ledgers[scale]
    dcc_spacing = ledger["dcc_pos"].diff()
    clock_records.append({
        "scale": scale.value,
        "threshold": SCALE_SETTINGS[scale]["theta"],
        "threshold_ratio_to_previous": (
            np.nan
            if scale == Scale.SMALL
            else SCALE_SETTINGS[scale]["theta"] / SCALE_SETTINGS[SCALE_ORDER[SCALE_ORDER.index(scale) - 1]]["theta"]
        ),
        "events": len(ledger),
        "median_t": ledger["t"].median(),
        "mean_t": ledger["t"].mean(),
        "median_tmv": ledger["tmv"].median(),
        "median_r": ledger["r"].median(),
        "median_bars_between_dccs": dcc_spacing.median(),
    })
clock_summary = pd.DataFrame.from_records(clock_records).set_index("scale")
clock_summary.style.format({
    "threshold": "{:.2%}",
    "threshold_ratio_to_previous": "{:.3f}",
    "events": "{:,}",
    "median_t": "{:.1f}",
    "mean_t": "{:.1f}",
    "median_tmv": "{:.3f}",
    "median_r": "{:.3e}",
    "median_bars_between_dccs": "{:.1f}",
})

,threshold,threshold_ratio_to_previous,events,median_t,mean_t,median_tmv,median_r,median_bars_between_dccs
scale,,,,,,,,
small,0.20%,nan,"2,697",60.0,95.8,1.858,6.655e-05,60.0
medium,0.40%,2.000,839,229.0,308.0,1.747,3.256e-05,227.5
large,0.60%,1.500,392,473.5,657.4,1.718,2.413e-05,448.0
macro,0.80%,1.333,220,797.5,1171.4,1.791,1.760e-05,792.0


Equal absolute spacing does not mean equal adjacent relationships. The ratios are 2.0, 1.5, and approximately 1.33. The upper edges therefore require progressively smaller additional movement relative to the already-confirmed lower threshold.

## Four completed-event overlays

Each scale has its own colour pair, line width, and dash pattern. The DCC triangle shows the new live mode; the connected EXT-to-EXT segment is the completed prior leg.

In [6]:
def direction_color(scale: Scale, direction: str) -> str:
    key = "up_color" if direction == DcMode.UP.value else "down_color"
    return SCALE_SETTINGS[scale][key]


def plot_four_threshold_events(close: pd.Series, ledgers: dict, n_prices: int) -> go.Figure:
    if n_prices <= 0:
        raise ValueError("n_prices must be positive.")
    price_window = close.tail(n_prices)
    window_start = price_window.index[0]
    window_end = price_window.index[-1]
    figure = go.Figure()
    figure.add_trace(go.Scattergl(
        x=price_window.index,
        y=price_window.to_numpy(),
        mode="lines",
        name="Close",
        line={"color": "#475569", "width": 1.0},
        hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
    ))

    for scale in SCALE_ORDER:
        settings = SCALE_SETTINGS[scale]
        events = ledgers[scale]
        visible = events.loc[
            events["ext_start_idx"].le(window_end) & events["dcc_idx"].ge(window_start)
        ]
        for completed_direction in DcMode:
            directional = visible.loc[visible["direction"].eq(completed_direction.value)]
            x_values = []
            y_values = []
            for event in directional.itertuples(index=False):
                x_values.extend([event.ext_start_idx, event.ext_end_idx, None])
                y_values.extend([event.ext_start_price, event.ext_end_price, None])
            if x_values:
                figure.add_trace(go.Scatter(
                    x=x_values,
                    y=y_values,
                    mode="lines",
                    name=(
                        f"{scale.value.title()} {settings['theta']:.2%} "
                        f"completed {completed_direction.value}"
                    ),
                    legendgroup=f"{scale.value}_{completed_direction.value}",
                    line={
                        "color": direction_color(scale, completed_direction.value),
                        "width": settings["line_width"],
                        "dash": settings["line_dash"],
                    },
                    opacity=0.82,
                    hoverinfo="skip",
                ))

            confirmations = directional.loc[
                directional["dcc_idx"].between(window_start, window_end)
            ]
            if confirmations.empty:
                continue
            live_direction = (
                DcMode.DOWN.value
                if completed_direction == DcMode.UP
                else DcMode.UP.value
            )
            symbol = "triangle-down" if live_direction == DcMode.DOWN.value else "triangle-up"
            figure.add_trace(go.Scatter(
                x=confirmations["dcc_idx"],
                y=confirmations["dcc_price"],
                mode="markers",
                name=f"{scale.value.title()} {live_direction} DCC",
                legendgroup=f"{scale.value}_{live_direction}",
                showlegend=False,
                marker={
                    "color": direction_color(scale, live_direction),
                    "size": 4 + SCALE_ORDER.index(scale),
                    "symbol": symbol,
                    "line": {"color": "white", "width": 0.6},
                },
                customdata=confirmations[[
                    "event_id", "tmv", "t", "r", "ext_end_idx", "information_time",
                ]].to_numpy(),
                hovertemplate=(
                    f"{scale.value.title()} {live_direction} DCC<br>"
                    "time=%{x}<br>price=%{y:.5f}<br>event=%{customdata[0]}<br>"
                    "TMV=%{customdata[1]:.3f}<br>T=%{customdata[2]}<br>"
                    "R=%{customdata[3]:.3e}<br>confirmed EXT=%{customdata[4]}<br>"
                    "known at=%{customdata[5]}<extra></extra>"
                ),
            ))

    figure.update_layout(
        title=f"{SYMBOL}: four independent DC clocks — last {len(price_window):,} prices",
        template="plotly_white",
        height=800,
        hovermode="closest",
        legend={"orientation": "h", "y": 1.12, "x": 0.0},
        margin={"l": 65, "r": 30, "t": 145, "b": 65},
    )
    figure.update_xaxes(title="Time", range=[window_start, window_end])
    figure.update_yaxes(title="Close")
    return figure


plot_four_threshold_events(prices, event_ledgers, N_PRICES).show()

## Construct the causal four-mode state path

In [7]:
from dc_basics.configuration import classify_configuration as _classify_configuration

def classify_configuration(modes: tuple[str, ...]) -> dict[str, object]:
    return _classify_configuration(modes, outer_scale_name="macro", expected_count=4)


configuration_reference = pd.DataFrame.from_records([
    classify_configuration(tuple(modes))
    for modes in product([DcMode.UP.value, DcMode.DOWN.value], repeat=len(SCALE_ORDER))
]).set_index("configuration")
configuration_reference = configuration_reference.reindex(CONFIGURATION_ORDER)
if len(configuration_reference) != 16 or configuration_reference.isna().all(axis=1).any():
    raise AssertionError("The 16-configuration reference is incomplete.")
configuration_reference

,hierarchy_shape,direction_boundaries,macro_agreement_breadth,small_coherence_depth,counter_depth,coherent_frontier
configuration,,,,,,
up/up/up/up,broad_agreement,0,4,4,0,True
up/up/up/down,counter_depth_3,1,1,3,3,True
up/up/down/up,fragmented_2_boundaries,2,3,2,<NA>,False
up/up/down/down,counter_depth_2,1,2,2,2,True
up/down/up/up,fragmented_2_boundaries,2,3,1,<NA>,False
up/down/up/down,fragmented_3_boundaries,3,2,1,<NA>,False
up/down/down/up,fragmented_2_boundaries,2,2,1,<NA>,False
up/down/down/down,counter_depth_1,1,3,1,1,True
down/up/up/up,counter_depth_1,1,3,1,1,True


In [8]:
mode_columns = [f"{scale.value}_mode" for scale in SCALE_ORDER]
all_modes_known = mode_frame[mode_columns].notna().all(axis=1)
known_modes = mode_frame.loc[all_modes_known, mode_columns]
property_records = [
    classify_configuration(tuple(row))
    for row in known_modes.itertuples(index=False, name=None)
]
properties = pd.DataFrame.from_records(property_records, index=known_modes.index)
for column in properties.columns:
    mode_frame.loc[properties.index, column] = properties[column]

required_properties = [
    "configuration", "hierarchy_shape", "direction_boundaries",
    "macro_agreement_breadth", "small_coherence_depth",
    "counter_depth", "coherent_frontier",
]
if mode_frame.loc[all_modes_known, required_properties].isna().drop(
    columns=["counter_depth"]
).any().any():
    raise AssertionError("A fully initialized bar lacks a hierarchy property.")

initialization_audit = pd.DataFrame({
    "first_mode_time": [live_frames[scale]["direction"].first_valid_index() for scale in SCALE_ORDER],
    "completed_events": [len(event_ledgers[scale]) for scale in SCALE_ORDER],
}, index=[scale.value for scale in SCALE_ORDER])
initialization_audit.loc["all_scales", "first_mode_time"] = mode_frame.index[all_modes_known][0]
initialization_audit

,first_mode_time,completed_events
small,2023-05-22 16:15:00+02:00,2697.0
medium,2023-05-22 17:50:00+02:00,839.0
large,2023-05-23 11:30:00+02:00,392.0
macro,2023-05-24 13:35:00+02:00,220.0
all_scales,2023-05-24 13:35:00+02:00,NaN


In [9]:
def plot_four_mode_ribbons(close: pd.Series, frame: pd.DataFrame, n_prices: int) -> go.Figure:
    price_window = close.tail(n_prices)
    modes = frame.loc[price_window.index]
    figure = make_subplots(
        rows=5,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.68, 0.08, 0.08, 0.08, 0.08],
        vertical_spacing=0.025,
        subplot_titles=["Close", "Small", "Medium", "Large", "Macro"],
    )
    figure.add_trace(go.Scattergl(
        x=price_window.index,
        y=price_window.to_numpy(),
        mode="lines",
        name="Close",
        line={"color": "#475569", "width": 1.0},
        hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
    ), row=1, col=1)

    for row_number, scale in enumerate(SCALE_ORDER, start=2):
        for direction in DcMode:
            ribbon = np.where(modes[f"{scale.value}_mode"].eq(direction.value), 1.0, np.nan)
            figure.add_trace(go.Scattergl(
                x=modes.index,
                y=ribbon,
                mode="lines",
                name=f"{scale.value.title()} {direction.value}",
                legendgroup=f"{scale.value}_{direction.value}_mode",
                line={"color": direction_color(scale, direction.value), "width": 12},
                connectgaps=False,
                hovertemplate=(
                    f"{scale.value.title()} mode={direction.value}<br>"
                    "time=%{x}<extra></extra>"
                ),
            ), row=row_number, col=1)
        figure.update_yaxes(showticklabels=False, range=[0.9, 1.1], row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: causal four-scale live modes — last {len(price_window):,} prices",
        template="plotly_white",
        height=900,
        hovermode="x unified",
        legend={"orientation": "h", "y": 1.08, "x": 0.0},
    )
    figure.update_xaxes(title="Time", row=5, col=1)
    return figure


plot_four_mode_ribbons(prices, mode_frame, N_PRICES).show()

## Six structural shapes from sixteen configurations

The six shapes are direction-neutral:

- broad agreement: all four modes agree;
- counter depth 1: only small opposes the upper three;
- counter depth 2: small and medium oppose large and macro;
- counter depth 3: small, medium, and large oppose macro;
- fragmented with two boundaries;
- fragmented with three boundaries.

Counter depth is defined only for a coherent single-front configuration. It is deliberately missing for fragmented states.

In [10]:
state_path = mode_frame.loc[all_modes_known, [*mode_columns, *required_properties]].copy()
state_path = state_path.reset_index(names="time")
state_path["bar_position"] = prices.index.get_indexer(state_path["time"])
state_path["run_id"] = (
    state_path["configuration"].ne(state_path["configuration"].shift()).fillna(True).cumsum()
)
state_runs = state_path.groupby("run_id", sort=True).agg(
    configuration=("configuration", "first"),
    hierarchy_shape=("hierarchy_shape", "first"),
    direction_boundaries=("direction_boundaries", "first"),
    macro_agreement_breadth=("macro_agreement_breadth", "first"),
    counter_depth=("counter_depth", "first"),
    start_time=("time", "first"),
    end_time=("time", "last"),
    start_position=("bar_position", "first"),
    end_position=("bar_position", "last"),
    bars=("configuration", "size"),
).reset_index(drop=True)

configuration_summary = state_runs.groupby("configuration", sort=False).agg(
    runs=("configuration", "size"),
    median_run_bars=("bars", "median"),
    p90_run_bars=("bars", lambda values: values.quantile(0.90)),
)
configuration_summary["bars"] = state_path["configuration"].value_counts()
configuration_summary = configuration_summary.reindex(CONFIGURATION_ORDER)
configuration_summary[["runs", "bars"]] = (
    configuration_summary[["runs", "bars"]].fillna(0).astype(int)
)
configuration_summary["bar_share"] = configuration_summary["bars"] / len(state_path)

shape_summary = state_runs.groupby("hierarchy_shape", sort=False).agg(
    runs=("hierarchy_shape", "size"),
    median_run_bars=("bars", "median"),
    p90_run_bars=("bars", lambda values: values.quantile(0.90)),
)
shape_summary["bars"] = state_path["hierarchy_shape"].value_counts()
shape_summary["bar_share"] = shape_summary["bars"] / len(state_path)

display(configuration_summary.style.format({
    "runs": "{:,}",
    "median_run_bars": "{:.1f}",
    "p90_run_bars": "{:.1f}",
    "bars": "{:,}",
    "bar_share": "{:.2%}",
}))
display(shape_summary.style.format({
    "runs": "{:,}",
    "median_run_bars": "{:.1f}",
    "p90_run_bars": "{:.1f}",
    "bars": "{:,}",
    "bar_share": "{:.2%}",
}))

,runs,median_run_bars,p90_run_bars,bars,bar_share
configuration,,,,,
up/up/up/up,509,59.0,227.2,"48,431",18.77%
up/up/up/down,241,30.0,136.0,"12,914",5.01%
up/up/down/up,89,37.0,123.2,"4,858",1.88%
up/up/down/down,328,33.5,153.2,"19,830",7.69%
up/down/up/up,194,36.0,156.6,"11,302",4.38%
up/down/up/down,57,28.0,120.4,"2,898",1.12%
up/down/down/up,129,31.0,127.8,"6,788",2.63%
up/down/down/down,488,39.0,150.2,"28,754",11.14%
down/up/up/up,492,37.0,159.0,"30,134",11.68%


,runs,median_run_bars,p90_run_bars,bars,bar_share
hierarchy_shape,,,,,
broad_agreement,"1,006",54.0,210.0,"87,414",33.88%
counter_depth_1,980,38.0,157.0,"58,888",22.82%
counter_depth_2,669,32.0,151.2,"38,597",14.96%
counter_depth_3,480,26.0,132.1,"23,654",9.17%
fragmented_2_boundaries,825,32.0,135.0,"43,806",16.98%
fragmented_3_boundaries,104,27.5,155.5,"5,643",2.19%


In [11]:
shape_order = [shape.value for shape in HierarchyShape]
shape_colors = {
    HierarchyShape.BROAD_AGREEMENT.value: "#334155",
    HierarchyShape.COUNTER_DEPTH_1.value: "#16a34a",
    HierarchyShape.COUNTER_DEPTH_2.value: "#2563eb",
    HierarchyShape.COUNTER_DEPTH_3.value: "#7c3aed",
    HierarchyShape.FRAGMENTED_2_BOUNDARIES.value: "#f59e0b",
    HierarchyShape.FRAGMENTED_3_BOUNDARIES.value: "#dc2626",
}
ordered_shapes = shape_summary.reindex(shape_order).dropna(subset=["bars"])
shape_figure = go.Figure(go.Bar(
    x=ordered_shapes.index,
    y=ordered_shapes["bar_share"],
    marker={"color": [shape_colors[shape] for shape in ordered_shapes.index]},
    customdata=ordered_shapes[["runs", "median_run_bars", "p90_run_bars"]].to_numpy(),
    hovertemplate=(
        "shape=%{x}<br>bar share=%{y:.2%}<br>runs=%{customdata[0]:,.0f}<br>"
        "median run=%{customdata[1]:.1f}<br>p90 run=%{customdata[2]:.1f}<extra></extra>"
    ),
))
shape_figure.update_layout(
    title="Causal occupancy of the six four-scale structural shapes",
    template="plotly_white",
    height=500,
    xaxis_title="Hierarchy shape",
    yaxis_title="Share of fully initialized bars",
    yaxis_tickformat=".0%",
)
shape_figure.show()

property_occupancy = pd.crosstab(
    state_path["direction_boundaries"],
    state_path["macro_agreement_breadth"],
    normalize="all",
)
property_occupancy.index.name = "direction boundaries"
property_occupancy.columns.name = "scales matching macro"
property_occupancy.style.format("{:.2%}")

scales matching macro,1.000000,2.000000,3.000000,4.000000
direction boundaries,,,,
0.000000,0.00%,0.00%,0.00%,33.88%
1.000000,9.17%,14.96%,22.82%,0.00%
2.000000,0.00%,5.30%,11.67%,0.00%
3.000000,0.00%,2.19%,0.00%,0.00%


## Configuration transitions and same-bar ambiguity

A component changes only when its own threshold confirms a DCC. When several components change on the same close bar, their intrabar order is unknowable from close-only data. We record the joint transition without inventing an ordering.

In [12]:
previous_configuration = state_path["configuration"].shift()
is_transition = state_path["configuration"].ne(previous_configuration) & previous_configuration.notna()
transition_columns = [
    "time", "bar_position", "configuration", "hierarchy_shape",
    "direction_boundaries", "macro_agreement_breadth", *mode_columns,
]
transitions = state_path.loc[is_transition, transition_columns].copy()
transitions["previous_configuration"] = previous_configuration.loc[is_transition].to_numpy()
transitions["previous_shape"] = state_path["hierarchy_shape"].shift().loc[is_transition].to_numpy()

changed_columns = []
for scale in SCALE_ORDER:
    mode_column = f"{scale.value}_mode"
    changed_column = f"{scale.value}_changed"
    transitions[changed_column] = (
        state_path.loc[is_transition, mode_column].to_numpy()
        != state_path[mode_column].shift().loc[is_transition].to_numpy()
    )
    changed_columns.append(changed_column)
    dcc_columns = [f"{scale.value}_upward_dcc", f"{scale.value}_downward_dcc"]
    transition_times = pd.DatetimeIndex(transitions["time"])
    observed_dcc = mode_frame.loc[transition_times, dcc_columns].any(axis=1).to_numpy()
    if not np.array_equal(transitions[changed_column].to_numpy(), observed_dcc):
        raise AssertionError(f"A {scale.value} mode transition disagrees with its DCC flags.")

transitions["changed_scale_count"] = transitions[changed_columns].sum(axis=1)
transitions["transition_signature"] = [
    "+".join(
        scale.value
        for scale, changed in zip(SCALE_ORDER, changed_flags, strict=True)
        if changed
    )
    for changed_flags in transitions[changed_columns].itertuples(index=False, name=None)
]
transition_summary = transitions.groupby("transition_signature", sort=False).agg(
    transitions=("configuration", "size"),
    median_boundaries_after=("direction_boundaries", "median"),
)
transition_summary["share"] = transition_summary["transitions"] / len(transitions)
shape_transition_matrix = pd.crosstab(
    transitions["previous_shape"],
    transitions["hierarchy_shape"],
    rownames=["from shape"],
    colnames=["to shape"],
)
display(transition_summary.style.format({
    "transitions": "{:,}",
    "median_boundaries_after": "{:.1f}",
    "share": "{:.2%}",
}))
display(shape_transition_matrix)

same_bar_transitions = transitions.loc[transitions["changed_scale_count"].gt(1)]
print(f"All configuration transitions: {len(transitions):,}")
print(f"Multi-scale same-bar transitions: {len(same_bar_transitions):,}")
print(f"Multi-scale share: {len(same_bar_transitions) / len(transitions):.2%}")

,transitions,median_boundaries_after,share
transition_signature,,,
small,"2,637",1.0,64.90%
medium,785,1.0,19.32%
large,366,1.0,9.01%
macro,209,0.0,5.14%
small+medium,36,1.0,0.89%
small+macro,5,0.0,0.12%
small+large,7,1.0,0.17%
small+medium+large,6,1.0,0.15%
medium+large,6,1.0,0.15%


to shape,broad_agreement,counter_depth_1,counter_depth_2,counter_depth_3,fragmented_2_boundaries,fragmented_3_boundaries
from shape,,,,,,
broad_agreement,2,980,20,4,0,0
counter_depth_1,532,0,442,6,0,0
counter_depth_2,9,0,0,283,376,0
counter_depth_3,211,0,0,0,269,0
fragmented_2_boundaries,250,0,207,135,129,104
fragmented_3_boundaries,1,0,0,52,51,0


All configuration transitions: 4,063
Multi-scale same-bar transitions: 66
Multi-scale share: 1.62%


## Three adjacent propagation contests

Each adjacent edge is still the familiar two-clock contest:

1. small → medium;
2. medium → large;
3. large → macro.

An attempt propagates if the upper DCC joins the lower target direction first. It is rejected if the lower clock flips back first. These three edge tables are independent building blocks; they are not yet one linked four-scale genealogy.

In [13]:
from dc_basics.hierarchy import build_adjacent_attempts


In [14]:
adjacent_pairs = (
    (Scale.SMALL, Scale.MEDIUM),
    (Scale.MEDIUM, Scale.LARGE),
    (Scale.LARGE, Scale.MACRO),
)
adjacent_attempts = pd.concat([
    build_adjacent_attempts(mode_frame, lower_scale, upper_scale)
    for lower_scale, upper_scale in adjacent_pairs
], ignore_index=True)
adjacent_attempts["outcome_information_time"] = adjacent_attempts["resolution_time"]
adjacent_attempts["outcome_is_hindsight_at_start"] = adjacent_attempts[
    "outcome_information_time"
].gt(adjacent_attempts["start_time"])

resolved_outcomes = [AttemptOutcome.PROPAGATED.value, AttemptOutcome.REJECTED.value]
resolved_attempts = adjacent_attempts.loc[
    adjacent_attempts["outcome"].isin(resolved_outcomes)
].copy()
clean_attempts = resolved_attempts.loc[resolved_attempts["resolution_bars"].gt(0)].copy()
if not clean_attempts["outcome_is_hindsight_at_start"].all():
    raise AssertionError("A clean adjacent outcome was already known at initiation.")

edge_records = []
for edge, all_edge_attempts in adjacent_attempts.groupby("edge", sort=False):
    edge_resolved = resolved_attempts.loc[resolved_attempts["edge"].eq(edge)]
    edge_clean = clean_attempts.loc[clean_attempts["edge"].eq(edge)]
    edge_records.append({
        "edge": edge,
        "all_attempts": len(all_edge_attempts),
        "same_bar_propagations": int(edge_resolved["resolution_bars"].eq(0).sum()),
        "clean_future_attempts": len(edge_clean),
        "all_resolved_propagation_rate": edge_resolved["outcome"].eq(
            AttemptOutcome.PROPAGATED.value
        ).mean(),
        "clean_propagation_rate": edge_clean["outcome"].eq(
            AttemptOutcome.PROPAGATED.value
        ).mean(),
        "median_clean_resolution_bars": edge_clean["resolution_bars"].median(),
    })
edge_summary = pd.DataFrame.from_records(edge_records).set_index("edge")
edge_summary.style.format({
    "all_attempts": "{:,}",
    "same_bar_propagations": "{:,}",
    "clean_future_attempts": "{:,}",
    "all_resolved_propagation_rate": "{:.2%}",
    "clean_propagation_rate": "{:.2%}",
    "median_clean_resolution_bars": "{:.1f}",
})

,all_attempts,same_bar_propagations,clean_future_attempts,all_resolved_propagation_rate,clean_propagation_rate,median_clean_resolution_bars
edge,,,,,,
small->medium,"1,768",44,"1,724",47.45%,46.11%,35.0
medium->large,616,15,600,63.74%,62.83%,72.0
large->macro,305,5,300,72.13%,71.67%,117.0


## Adjacent event-count ratios and nesting

For alternating directional events, the lower-to-upper event-count ratio provides a rough mechanical guide to propagation frequency:

    approximate propagation rate = 2 / (event-count ratio + 1)

This is not a forecast. It describes how the two event clocks relate over the completed sample.

Between consecutive upper DCCs, the adjacent lower DCC count is positive and odd. Each pair before the terminal odd event represents one lower-scale rejection within the upper interval.

In [15]:
from dc_basics.hierarchy import build_nesting_intervals


nesting_intervals = pd.concat([
    build_nesting_intervals(
        event_ledgers[lower_scale], event_ledgers[upper_scale], lower_scale, upper_scale
    )
    for lower_scale, upper_scale in adjacent_pairs
], ignore_index=True)

nesting_summary = nesting_intervals.groupby("edge", sort=False).agg(
    upper_intervals=("upper_event_id", "size"),
    lower_dcc_count_median=("lower_dcc_count", "median"),
    lower_dcc_count_p90=("lower_dcc_count", lambda values: values.quantile(0.90)),
    prior_rejections_median=("prior_rejections", "median"),
    prior_rejections_p90=("prior_rejections", lambda values: values.quantile(0.90)),
    interval_bars_median=("interval_bars", "median"),
)
nesting_summary.style.format({
    "upper_intervals": "{:,}",
    "lower_dcc_count_median": "{:.1f}",
    "lower_dcc_count_p90": "{:.1f}",
    "prior_rejections_median": "{:.1f}",
    "prior_rejections_p90": "{:.1f}",
    "interval_bars_median": "{:.1f}",
})

,upper_intervals,lower_dcc_count_median,lower_dcc_count_p90,prior_rejections_median,prior_rejections_p90,interval_bars_median
edge,,,,,,
small->medium,838,3.0,7.0,1.0,3.0,227.5
medium->large,391,1.0,5.0,0.0,2.0,448.0
large->macro,219,1.0,3.0,0.0,1.0,792.0


In [16]:
edge_ratio_records = []
for lower_scale, upper_scale in adjacent_pairs:
    edge = f"{lower_scale.value}->{upper_scale.value}"
    event_count_ratio = len(event_ledgers[lower_scale]) / len(event_ledgers[upper_scale])
    edge_ratio_records.append({
        "edge": edge,
        "lower_threshold": SCALE_SETTINGS[lower_scale]["theta"],
        "upper_threshold": SCALE_SETTINGS[upper_scale]["theta"],
        "threshold_ratio": (
            SCALE_SETTINGS[upper_scale]["theta"] / SCALE_SETTINGS[lower_scale]["theta"]
        ),
        "event_count_ratio": event_count_ratio,
        "approximate_propagation_rate": 2.0 / (event_count_ratio + 1.0),
        "observed_all_resolved_rate": edge_summary.loc[
            edge, "all_resolved_propagation_rate"
        ],
        "observed_clean_future_rate": edge_summary.loc[edge, "clean_propagation_rate"],
    })
edge_ratio_summary = pd.DataFrame.from_records(edge_ratio_records).set_index("edge")
edge_ratio_summary.style.format({
    "lower_threshold": "{:.2%}",
    "upper_threshold": "{:.2%}",
    "threshold_ratio": "{:.3f}",
    "event_count_ratio": "{:.3f}",
    "approximate_propagation_rate": "{:.2%}",
    "observed_all_resolved_rate": "{:.2%}",
    "observed_clean_future_rate": "{:.2%}",
})

,lower_threshold,upper_threshold,threshold_ratio,event_count_ratio,approximate_propagation_rate,observed_all_resolved_rate,observed_clean_future_rate
edge,,,,,,,
small->medium,0.20%,0.40%,2.000,3.215,47.45%,47.45%,46.11%
medium->large,0.40%,0.60%,1.500,2.140,63.69%,63.74%,62.83%
large->macro,0.60%,0.80%,1.333,1.782,71.90%,72.13%,71.67%


## Structural-shape price casebook

The most recent run of each observed shape is selected with hindsight for teaching. The shaded band is the shape's causal occupancy interval. DCC markers show which clock changed near the example.

In [17]:
shape_examples = (
    state_runs.sort_values("start_time")
    .groupby("hierarchy_shape", sort=True, group_keys=False)
    .tail(1)
    .sort_values("hierarchy_shape")
)
display(shape_examples[[
    "hierarchy_shape", "configuration", "start_time", "end_time",
    "bars", "direction_boundaries", "macro_agreement_breadth", "counter_depth",
]])

,hierarchy_shape,configuration,start_time,end_time,bars,direction_boundaries,macro_agreement_breadth,counter_depth
4061,broad_agreement,down/down/down/down,2026-11-06 21:45:00+02:00,2026-11-09 03:10:00+02:00,66,0.0,4.0,0
4062,counter_depth_1,up/down/down/down,2026-11-09 03:15:00+02:00,2026-11-09 10:00:00+02:00,82,1.0,3.0,1
4063,counter_depth_2,up/up/down/down,2026-11-09 10:05:00+02:00,2026-11-09 13:30:00+02:00,42,1.0,2.0,2
4058,counter_depth_3,down/down/down/up,2026-11-06 08:45:00+02:00,2026-11-06 11:00:00+02:00,28,1.0,1.0,3
4046,fragmented_2_boundaries,up/down/up/up,2026-10-30 19:10:00+02:00,2026-11-02 10:30:00+02:00,185,2.0,3.0,None
3936,fragmented_3_boundaries,up/down/up/down,2026-09-11 15:00:00+02:00,2026-09-14 05:25:00+02:00,173,3.0,2.0,None


In [18]:
def plot_shape_casebook(close: pd.Series, examples: pd.DataFrame) -> go.Figure:
    figure = make_subplots(
        rows=len(examples),
        cols=1,
        vertical_spacing=min(0.045, 0.16 / len(examples)),
        subplot_titles=[
            f"{example.hierarchy_shape} | {example.configuration} | {example.bars} bars"
            for example in examples.itertuples(index=False)
        ],
    )

    for row_number, example in enumerate(examples.itertuples(index=False), start=1):
        window_start_position = max(0, int(example.start_position) - CASEBOOK_BARS_BEFORE)
        window_end_position = min(
            len(close), int(example.end_position) + CASEBOOK_BARS_AFTER + 1
        )
        price_window = close.iloc[window_start_position:window_end_position]
        active_prices = close.iloc[int(example.start_position):int(example.end_position) + 1]
        figure.add_trace(go.Scattergl(
            x=price_window.index,
            y=price_window.to_numpy(),
            mode="lines",
            name="Close",
            legendgroup="close",
            showlegend=row_number == 1,
            line={"color": "#94a3b8", "width": 1.0},
            hovertemplate="time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)
        figure.add_trace(go.Scattergl(
            x=active_prices.index,
            y=active_prices.to_numpy(),
            mode="lines",
            name="Selected causal shape run",
            legendgroup="active_shape",
            showlegend=row_number == 1,
            line={"color": shape_colors[example.hierarchy_shape], "width": 3.0},
            hovertemplate="active shape<br>time=%{x}<br>close=%{y:.5f}<extra></extra>",
        ), row=row_number, col=1)

        for scale in SCALE_ORDER:
            ledger = event_ledgers[scale]
            visible_dccs = ledger.loc[
                ledger["dcc_idx"].between(price_window.index[0], price_window.index[-1])
            ]
            if visible_dccs.empty:
                continue
            figure.add_trace(go.Scatter(
                x=visible_dccs["dcc_idx"],
                y=visible_dccs["dcc_price"],
                mode="markers",
                name=f"{scale.value.title()} DCC",
                legendgroup=f"{scale.value}_dcc",
                showlegend=row_number == 1,
                marker={
                    "color": SCALE_SETTINGS[scale]["up_color"],
                    "size": 4 + SCALE_ORDER.index(scale),
                    "symbol": "x",
                },
                customdata=visible_dccs[["event_id", "direction"]].to_numpy(),
                hovertemplate=(
                    f"{scale.value.title()} DCC<br>time=%{{x}}<br>price=%{{y:.5f}}<br>"
                    "completed event=%{customdata[0]}<br>"
                    "completed direction=%{customdata[1]}<extra></extra>"
                ),
            ), row=row_number, col=1)
        figure.update_yaxes(title_text="Close", row=row_number, col=1)

    figure.update_layout(
        title=f"{SYMBOL}: recent four-scale structural-shape casebook",
        template="plotly_white",
        height=max(900, 290 * len(examples)),
        hovermode="closest",
        legend={"orientation": "h", "y": 1.05, "x": 0.0},
    )
    return figure


plot_shape_casebook(prices, shape_examples).show()

# Four-threshold mechanics completion gate

Before linking a small-origin disturbance through all four scales, be able to explain:

1. Why four clocks remain independent despite sharing one price path.
2. How to read every configuration in small / medium / large / macro order.
3. Why down/down/up/up is a coherent depth-2 counter-move rather than fragmentation.
4. Why down/up/down/down is fragmented even though three modes point down.
5. Why the number of scales matching macro does not uniquely determine structural shape.
6. Why coherent counter depth is undefined for configurations with multiple boundaries.
7. Which exact DCC changes each configuration component.
8. Why same-bar multi-scale transitions have unknowable intrabar ordering.
9. Why equally spaced absolute thresholds have unequal adjacent ratios.
10. How those ratios influence adjacent event counts and propagation frequencies.
11. How many lower DCCs nest inside each adjacent upper interval.
12. Why nested counts are odd and how prior rejection pairs are encoded.
13. Why adjacent edge outcomes are hindsight until their resolution DCC.
14. Why the three edge tables are not yet one linked four-scale lineage.
15. What broad coherence, shallow disturbance, deep propagation, and fragmentation look like on price.

The next step is four-scale linkage: pin one broad-agreement small seed and follow exact event-ID hand-offs through medium, large, and macro. Live development and classification remain downstream.